In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# BLIP-2 Zero-Shot Evaluation — Pics Can Lie

**Model:** `Salesforce/blip2-opt-2.7b` (pretrained, no fine-tuning)  
**Task:** Classify image-caption pairs as `real` (0) or `out-of-context` (1)  
**Dataset:** `D:/Pics Can Lie/merged_balanced/train.json`

## 1 — Config

In [1]:
import json
import os
import random

import torch
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import AutoProcessor, Blip2ForConditionalGeneration

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID       = "Salesforce/blip2-opt-2.7b"
ANNOTATIONS    = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'data', 'NewsClipPings', 'merged_balanced', 'train.json')
METADATA       = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'data', 'NewsClipPings', 'metadata', 'train.json')
IMAGES_ROOT    = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset', 'origin')
OUTPUT_PATH    = _os.path.join(str(_cfg.ROOT), 'results', 'blip2_pretrained_eval.json')
MAX_SAMPLES    = 500
SEED           = 42
MAX_NEW_TOKENS = 20

print("Config loaded.")

d:\Pics Can Lie\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config loaded.


## 2 — Load Dataset

In [ ]:
with open(ANNOTATIONS, "r", encoding="utf-8") as f:
    annotations = json.load(f)["annotations"]

with open(METADATA, "r", encoding="utf-8") as f:
    metadata = json.load(f)  # keyed by image_id (string)

def resolve_image_path(meta_image_path: str) -> str:
    # strip leading 'visual_news/' then join with IMAGES_ROOT
    rel = meta_image_path.replace("visual_news/", "", 1)
    return os.path.join(IMAGES_ROOT, rel)

# Join annotations with metadata
joined = []
for ann in annotations:
    image_id = str(ann["image_id"])
    if image_id not in metadata:
        continue
    meta    = metadata[image_id]
    caption = meta.get("caption") or meta.get("title") or ""
    joined.append({
        "image_path": resolve_image_path(meta["image_path"]),
        "caption":    caption,
        "label":      int(ann["falsified"]),
    })

random.seed(SEED)
samples = random.sample(joined, min(MAX_SAMPLES, len(joined)))

real_count = sum(1 for s in samples if s["label"] == 0)
ooc_count  = sum(1 for s in samples if s["label"] == 1)
print(f"Loaded {len(samples)} samples  |  real: {real_count}  out-of-context: {ooc_count}")

Loaded 500 samples  |  real: 250  out-of-context: 250


: 

## 3 — Load Model & Processor

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Device: {device}  |  dtype: {dtype}")

print("Loading processor …")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model …")
model = Blip2ForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=dtype)
model.to(device)
model.eval()
print("Model ready.")

Device: cuda  |  dtype: torch.float16
Loading processor …


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading model …


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]d:\Pics Can Lie\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Youssef Elghandour\.cache\huggingface\hub\models--Salesforce--blip2-opt-2.7b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 2 files: 100%|██████████| 2/2 [39:37<00:00, 1

## 4 — Run Inference

In [ ]:
def classify_response(response: str) -> int:
    """Return 1 (out-of-context) if response signals mismatch, else 0 (real)."""
    lower = response.lower().strip()
    if any(kw in lower for kw in ("out-of-context", "no", "false")):
        return 1
    return 0


results = []
y_true  = []
y_pred  = []
skipped = 0

for i, item in enumerate(samples, start=1):
    try:
        image = Image.open(item["image_path"]).convert("RGB")
    except Exception as e:
        print(f"[{i}/{len(samples)}] SKIP — {e}")
        skipped += 1
        continue

    prompt = (
        f"Does this image match the caption: '{item['caption'][:100]}'? "
        "Answer:"
    )

    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device, dtype)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

    response  = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
    predicted = classify_response(response)

    y_true.append(item["label"])
    y_pred.append(predicted)
    results.append({
        "image_path": item["image_path"],
        "caption":    item["caption"],
        "label":      item["label"],
        "response":   response,
        "predicted":  predicted,
        "correct":    predicted == item["label"],
    })

    if i % 50 == 0 or i == len(samples):
        running_acc = accuracy_score(y_true, y_pred)
        print(f"[{i}/{len(samples)}]  running accuracy: {running_acc:.3f}")

print(f"\nDone. Evaluated: {len(y_true)}  Skipped: {skipped}")

## 5 — Metrics

In [ ]:
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)
cm   = confusion_matrix(y_true, y_pred).tolist()

print("=" * 60)
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1       : {f1:.4f}")
print(f"Skipped  : {skipped}")
print("\nConfusion matrix (rows=true, cols=pred):")
print(f"  TN={cm[0][0]}  FP={cm[0][1]}")
print(f"  FN={cm[1][0]}  TP={cm[1][1]}")
print("\nClassification report:")
print(classification_report(y_true, y_pred,
                            target_names=["real", "out-of-context"],
                            zero_division=0))

## 6 — Save Results

In [ ]:
output = {
    "config": {
        "model":       MODEL_ID,
        "annotations": ANNOTATIONS,
        "max_samples": MAX_SAMPLES,
        "seed":        SEED,
        "evaluated":   len(y_true),
        "skipped":     skipped,
    },
    "metrics": {
        "accuracy":         acc,
        "precision":        prec,
        "recall":           rec,
        "f1":               f1,
        "confusion_matrix": cm,
    },
    "predictions": results,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Results saved → {OUTPUT_PATH}")